# Phase 3A - Autoresearch loop & group split

**Phase 3A - The autoresearch loop & why we split into groups.** Independent notebook - runs standalone in Colab or locally.

Loop: agent edits `train.py` -> 5-min train -> records `val_bpb` -> keep/revise/discard -> repeat. Memory-debug: remember which configs OOM/regressed so the agent avoids them. A single full sweep hits CUDA OOM at depth>=12, so we split into groups (deeper models use a smaller batch):

| Group | configs (depth, batch) |
|---|---|
| A | (4,32),(6,32) |
| B | (8,32),(10,32) |
| C | (12,16),(14,16) |
| D | (16,16),(18,16) |

## 0. Setup (self-contained)

In [ ]:
# Self-contained setup - works standalone in Google Colab or locally.
import sys, os, subprocess
from pathlib import Path
REPO_URL = "https://github.com/syaikhipin/kdd26-memdiag" # change in scripts/build_phase_notebooks.py to retarget everywhere
try:
 import google.colab # noqa
 IN_COLAB = True
except Exception:
 IN_COLAB = False
if IN_COLAB:
 repo = Path("/content/kdd26-memdiag")
 if not repo.exists():
 subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(repo)], check=False)
 SOURCE = repo / "experiment" / "github_submission" / "source"
 subprocess.run([sys.executable, "-m", "pip", "install", "-q", "numpy", "matplotlib", "pyyaml"], check=False)
else:
 SOURCE = None
 for cand in [Path.cwd(), *Path.cwd().parents]:
 for sub in ("source", "experiment"):
 if (cand / sub / "run.py").exists():
 SOURCE = cand / sub
 break
 if SOURCE:
 break
 if SOURCE is None:
 raise FileNotFoundError("Run from the repo root (or in Colab it auto-clones).")
sys.path.insert(0, str(SOURCE))
os.environ.setdefault("OPENAI_BASE_URL", "https://api.openai.com/v1")
SOURCE_DIR = SOURCE
PROJECT_ROOT = SOURCE.parent
RESULTS_DIR = PROJECT_ROOT / "results"
print("SOURCE_DIR =", SOURCE, "| IN_COLAB =", IN_COLAB)
